In [0]:
%sql
use catalog `p&l_silver`

In [0]:

# {path}/src/foodquest_pnl.py
from pyspark.sql.functions import *
import re, time
from pyspark.sql.window import Window

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df

def create_total_row(df, group_cols, total_col_name, total_label, dimension_cols):
    """
    Create aggregated total rows (subgroup, group, major group totals).
    
    Parameters:
    - df: Input dataframe
    - group_cols: List of columns to group by (e.g., ['location', 'month', 'year', 'sub_group'])
    - total_col_name: Column name to use for the total label (e.g., 'sub_group')
    - total_label: Label type ('Total' or 'Grand Total')
    - dimension_cols: List of dimension columns to preserve
    """
    return df.groupBy(*group_cols).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        # lit(None).alias("account_number"),
        concat(lit("Total "), col(total_col_name)).alias("account_name"),
        # concat(lit("Total "),col(total_col_name)).alias("name"),
        lit(None).alias("mapped_name"),
        lit(None).cast("string").alias("account_type"),
        lit(None).alias("major_group"),
        lit(None).alias("group"),
        lit(None).alias("sub_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit(total_label).alias("Detail/Total")
    )


def create_calculated_metric(df, metric_name, calculation_expr, dimension_cols, total_label="Grand Total"):
    """
    Create calculated metric rows (Gross Profit, Operating Profit, EBITDA, Net Profit).
    
    Parameters:
    - df: Input dataframe
    - metric_name: Name of the metric (e.g., 'Gross Profit')
    - calculation_expr: PySpark column expression for the calculation
    - dimension_cols: List of dimension columns to preserve
    - total_label: Label type (default 'Grand Total')
    """
    group_cols = ["location", "month", "year"] + dimension_cols
    
    return df.groupBy(*group_cols).agg(
        calculation_expr.alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        lit(metric_name).alias("account_name"),
        lit(None).alias("mapped_name"),
        lit(None).cast("string").alias("account_type"),
        lit(None).alias("major_group"),
        lit(None).alias("group"),
        lit(None).alias("sub_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit(total_label).alias("Detail/Total")
    )


def add_previous_year_data(df):
    """Add previous year (PY) amounts via self-join."""
    df_current = df.withColumn("year", col("year"))
    
    df_py = df_current.alias("py").select(
        (col("year") + 1).alias("year_join"),
        col("location").alias("py_location"),
        col("account_name").alias("py_account_name"),
        col("amount").alias("py_amount"),
        col("month").alias("py_month")
    )
    
    return df_current.alias("curr").join(
        df_py,
        (col("curr.year") == col("year_join")) &  
        (col("curr.location") == col("py_location")) &  
        (col("curr.account_name") == col("py_account_name")) &  
        (col("curr.month") == col("py_month")),
        "left"
    ).select(
        col("curr.*"),
        coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
    )


def add_net_sales_calculations(df):
    """Add actual and PY net sales calculations at location, brand, and company levels."""
    window_location = Window.partitionBy("location", "year", "month")
    window_brand = Window.partitionBy("brand_id", "year", "month")
    window_company = Window.partitionBy("company_id", "year", "month")
    
    sales_condition = col("account_name").like("Total Sales")
    
    return df \
        .withColumn("store_actual_net_sales",
            sum(when(sales_condition, col("amount")).otherwise(0.0)).over(window_location)) \
        .withColumn("store_py_net_sales",
            sum(when(sales_condition, col("py_amount")).otherwise(0.0)).over(window_location)) \
        .withColumn("brand_act_net_sales",
            sum(when(sales_condition, col("amount")).otherwise(0.0)).over(window_brand)) \
        .withColumn("brand_py_net_sales",
            sum(when(sales_condition, col("py_amount")).otherwise(0.0)).over(window_brand)) \
        .withColumn("company_act_net_sales",
            sum(when(sales_condition, col("amount")).otherwise(0.0)).over(window_company)) \
        .withColumn("company_py_net_sales",
            sum(when(sales_condition, col("py_amount")).otherwise(0.0)).over(window_company))


def get_dimension_columns():
    """Return list of standard dimension columns used throughout transformations."""
    return [
        "netsuite_location_name", "type", "location_id", "brand_id",
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ]

def join_dataframes(base_df, join_configs):
    """
    Perform multiple joins on a base dataframe.
    
    Parameters:
    -----------
    base_df : DataFrame
        The base dataframe to join other dataframes to
    join_configs : list of dict
        List of join configurations, each containing:
        - 'df': DataFrame to join
        - 'left_key': column name or expression from left/base df
        - 'right_key': column name or expression from right df
        - 'join_type': 'inner', 'left', 'right', 'outer', etc.
        
    Returns:
    --------
    DataFrame: Result of all joins applied sequentially
    
    Example:
    --------
    join_configs = [
        {
            'df': df_coa_master,
            'left_key': col("accountNumber").cast("string"),
            'right_key': df_coa_master["account_number"].cast("string"),
            'join_type': 'inner'
        },
        {
            'df': df_location_master,
            'left_key': col("location"),
            'right_key': df_location_master["netsuite_location_name"],
            'join_type': 'left'
        }
    ]
    result = join_dataframes(df, join_configs)
    """
    result_df = base_df
    
    for config in join_configs:
        result_df = result_df.join(
            config['df'],
            config['left_key'] == config['right_key'],
            config['join_type']
        )
    
    return result_df

def final_df(df_all_master, amount_type):
    # Load master tables
    # df_coa_master = spark.read.table("silver.dim_coa_master")
    # df_location_master = spark.read.table("silver.dim_location_master")
    
    
    
    # # Step 1: Join both master tables
    # df_all_masters = df.join(
    #     df_coa_master, 
    #     df_coa_master["account_number"].cast("string") == df["accountNumber"], 
    #     'inner'
    # ).join(
    #     df_location_master,
    #     col("location") == df_location_master.netsuite_location_name,
    #     'left'
    # )

    dimension_cols = get_dimension_columns()
    
    # Step 2: Create detail rows
    df_detail = df_all_masters.select(
        col("location"), col("month"), col("year"),
        col("account_name"), col("mapped_name"), col("account_type"),
        col("major_group"), col("group"), col("sub_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit("Detail").alias("Detail/Total")
    )
    
    # Step 3: Create total rows at different aggregation levels
    df_subgroup_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "major_group", "group", "sub_group"] + dimension_cols,
        "sub_group",
        "Total",
        dimension_cols
    )
    
    df_group_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "major_group", "group"] + dimension_cols,
        "group",
        "Total",
        dimension_cols
    )
    
    df_major_group_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "major_group"] + dimension_cols,
        "major_group",
        "Grand Total",
        dimension_cols
    )
    
    # Step 4: Create calculated metrics
    df_gross_profit = create_calculated_metric(
        df_all_masters,
        "Gross Profit",
        abs(sum(when(col("major_group") == "Sales", col("amount")).otherwise(0))) - 
        abs(sum(when(col("major_group") == "Purchases", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_operating_profit = create_calculated_metric(
        df_all_masters,
        "Operating Profit",
        abs(sum(when(col("major_group") == "Sales", col("amount")).otherwise(0))) - 
        abs(sum(when(col("major_group") == "Purchases", col("amount")).otherwise(0))) -
        abs(sum(when(col("major_group") == "Overheads", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_ebitda = create_calculated_metric(
        df_all_masters,
        "EBITDA",
        (abs(sum(when(col("major_group") == "Sales", col("amount")).otherwise(0))) - 
         abs(sum(when(col("major_group") == "Purchases", col("amount")).otherwise(0))) -
         abs(sum(when(col("major_group") == "Overheads", col("amount")).otherwise(0)))) +
        abs(sum(when(col("major_group") == "Depreciation & Amortization", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_net_profit = create_calculated_metric(
        df_all_masters,
        "Net Profit/(Loss)",
        (abs(sum(when(col("major_group") == "Sales", col("amount")).otherwise(0))) - 
         abs(sum(when(col("major_group") == "Purchases", col("amount")).otherwise(0))) -
         abs(sum(when(col("major_group") == "Overheads", col("amount")).otherwise(0)))) -
        abs(sum(when(col("major_group") == "Finance Costs", col("amount")).otherwise(0))) -
        abs(sum(when(col("major_group") == "Tax", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    # Step 5: Union all dataframes
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_major_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))
    
    if amount_type == 'actual':
            # Step 6: Add previous year data
        df_with_py = add_previous_year_data(df_combined)
        
        # Step 7: Calculate net sales metrics
        df_with_calculations = add_net_sales_calculations(df_with_py)
        
        # Step 8: Select final columns and convert to snake case
        df_final = df_with_calculations.select(
            col("account_name"), col("mapped_name"), col("account_type"),
            col("major_group"), col("group"), col("sub_group"),
            *[col(c) for c in dimension_cols],
            col("Detail/Total"), col("amount"), col("year"), col("month"),
            col("store_actual_net_sales"), col("store_py_net_sales"),
            col("brand_act_net_sales"), col("brand_py_net_sales"),
            col("company_act_net_sales"), col("company_py_net_sales")
        )
        
        df_final = to_snake_case_df(df_final)

        # Step 9: Join with sort order and final selection
        df_sort = spark.read.table('default.dim_coa_master')
        df_sort = to_snake_case_df(df_sort)
        df_sort = df_sort.select("account_name", "sort_order", "calculation_type")
        df_final_sort = df_final.join(
            df_sort, 
            df_sort["account_name"].cast("string") == df_final.account_name,
            'inner'
        ).drop(df_sort["account_name"]).select(
            col("account_name"), col("mapped_name"),
            col("major_group"), col("group"), col("sub_group"), col("account_type"),
            col("netsuite_location_name"), col("type"), col("location_id"),
            col("brand_id"), col("company_id"), col("store_type"),
            col("parent_company"), col("country_code"), col('zone'), col("city"),
            col("amount").alias(f"{amount_type}_amount"), col("year"), col("month"),
            col("store_actual_net_sales").alias(f"store_{amount_type}_net_sales"), col("store_py_net_sales").alias(f"store_py_net_sales"),
            col("brand_act_net_sales").alias(f"brand_{amount_type}_net_sales"), col("brand_py_net_sales").alias(f"brand_py_net_sales"),
            col("company_act_net_sales").alias(f"company_{amount_type}_net_sales"), col("company_py_net_sales").alias(f"company_py_net_sales"),
            col("detail/total"), col('sort_order'), col('calculation_type')
        )
    else:
        # Step 6: Add previous year data
        df_with_py = add_previous_year_data(df_combined)
        
        # Step 7: Calculate net sales metrics
        df_with_calculations = add_net_sales_calculations(df_with_py)
        # Step 8: Select final columns and convert to snake case
        df_final = df_with_calculations.select(
            col("account_name"), col("mapped_name"), col("account_type"),
            col("major_group"), col("group"), col("sub_group"),
            *[col(c) for c in dimension_cols],
            col("Detail/Total"), col("amount"), col("year"), col("month"),
            col("store_actual_net_sales"), col("store_py_net_sales"),
            col("brand_act_net_sales"), col("brand_py_net_sales"),
            col("company_act_net_sales"), col("company_py_net_sales")
        )
        
        df_final = to_snake_case_df(df_final)
        # Step 9: Join with sort order and final selection
        df_sort = spark.read.table('default.dim_coa_master')
        df_sort = to_snake_case_df(df_sort)
        df_sort = df_sort.select("account_name", "sort_order", "calculation_type")
        df_final_sort = df_final.join(
            df_sort, 
            df_sort["account_name"].cast("string") == df_final.account_name,
            'inner'
        ).drop(df_sort["account_name"]).select(
            col("account_name"), col("mapped_name"),
            col("major_group"), col("group"), col("sub_group"), col("account_type"),
            col("netsuite_location_name"), col("type"), col("location_id"),
            col("brand_id"), col("company_id"), col("store_type"),
            col("parent_company"), col("country_code"), col('zone'), col("city"),
            col("amount").alias(f"{amount_type}_amount"), col("year"), col("month"),
            col("store_actual_net_sales").alias(f"store_{amount_type}_net_sales"),
            col("brand_act_net_sales").alias(f"brand_{amount_type}_net_sales"),
            col("company_act_net_sales").alias(f"company_{amount_type}_net_sales"),
            col("detail/total"), col('sort_order'), col('calculation_type')
        )
    
    return df_final_sort.orderBy(
        "parent_company", "company_id", "brand_id", 
        "netsuite_location_name", 'year', 'month', 'sort_order'
    )

print("All variables and functions from other-notebook are now available")


In [0]:
df = spark.read.table('default.pnl_actual_flat_data')
df.select("Year").distinct().orderBy("Year").display()

In [0]:

df = spark.read.table('default.pnl_actual_flat_data')

from pyspark.sql.functions import *
df_filtered = df.filter((col("Store Name").isNotNull())
                        # | ((col("Store Name")=="Dubai Mall") & (col("File Name")=="January") & (col("Year")==2026)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2025)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2026))
)

df_coa_master = spark.read.table("default.dim_coa_master")
df_coa_master = to_snake_case_df(df_coa_master)
df_location_master = spark.read.table("default.dim_location_master")
df_location_master = to_snake_case_df(df_location_master)
df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping


# Step 1: Joined both master tables upfront
df_all_masters = df_filtered.join(
    df_coa_master_distinct, 
    df_coa_master_distinct["mapped_name"].cast("string") == df_filtered["Column 1"], 
    'inner'
).join(
    df_location_master,
    col("Store name") == df_location_master["excel_p&l_name"],
    'left'
)

df_all_masters = df_all_masters.withColumnsRenamed(
    {
        "File name": "month",
        "Year": "year",
        # "Column 1": "account_name",
        "Act": "amount",
        "Store name": "location"
    }
).withColumn(
    "amount",
    when(col("sub_group") == "Discount", -abs(col("amount")))
    .otherwise(col("amount"))
)

df_final_legacy_actual = final_df(df_all_masters, "actual")
df_final_legacy_actual.display()

In [0]:
df = spark.read.table('default.pnl_budget_flat_data')

from pyspark.sql.functions import *
df_filtered = df.filter((col("Store Name").isNotNull()))
                        #  | ((col("Store Name")=="Dubai Mall") & (col("File Name")=="January") & (col("Year")==2026)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2025)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2026)))
# df_filtered.display()

# df_coa_master = spark.read.table("default.dim_coa_master")
# df_location_master = spark.read.table("default.dim_location_master")
# df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping

# Step 1: Join both master tables upfront
df_all_masters = df_filtered.join(
    df_coa_master_distinct, 
    df_coa_master_distinct["mapped_name"].cast("string") == df_filtered["Column 1"], 
    'inner'
).join(
    df_location_master,
    col("Store name") == df_location_master["excel_p&l_name"],
    'left'
)

df_all_masters = df_all_masters.withColumnsRenamed(
{
    "File name": "month",
    "Year": "year",
    # "Column 1": "account_name",
    "Bud": "amount",
    "Store name": "location"
}
)


df_final_budget = final_df(df_all_masters, "budget")
df_final_budget.display()

In [0]:
from functools import reduce
from pyspark.sql.functions import col

join_keys = ['account_name', 'mapped_name','major_group', 'group', 'sub_group', 'account_type', 
             'netsuite_location_name', 'type', 'location_id', 'brand_id', 'company_id', 
             'store_type', 'parent_company', 'country_code', 'zone', 'city', 'year', 
             'month', 'detail/total', 'sort_order', 'calculation_type']

# Build null-safe join condition
join_condition = reduce(
    lambda a, b: a & b,
    [df_final_legacy_actual[k].eqNullSafe(df_final_budget[k]) for k in join_keys]
)

df_final_all = df_final_legacy_actual.join(df_final_budget, join_condition, 'outer')

# Drop duplicate key columns from the budget side
for k in join_keys:
    df_final_all = df_final_all.drop(df_final_budget[k])

# Fill null amounts with 0
df_final_all = df_final_all.fillna(0, subset=["actual_amount", "store_actual_net_sales", "store_py_net_sales", "budget_amount", "brand_actual_net_sales", "company_actual_net_sales","store_budget_net_sales", "brand_budget_net_sales", "company_budget_net_sales"])

df_final_all.display()

df_final_with_forecast = df_final_all.filter(col('netsuite_location_name').isNotNull()).withColumn(
    "final_forecast_amount",
    when(col("actual_amount") != 0, col("actual_amount"))
    .when(col("budget_amount") != 0, col("budget_amount"))
    .otherwise(lit(0.0))
).orderBy(
        "parent_company", "company_id", "brand_id", 
        "netsuite_location_name", 'year', 'month', 'sort_order'
)

df_final_with_forecast.display()

In [0]:
df_final_with_forecast.filter(((col('actual_amount') != 0) | (col('budget_amount') != 0)) & (col('year') == 2026) & (col('month') == 'February')).display()